# Mirai Stage1 Static Analysis Walkthrough

This notebook runs the local static pipeline and summarizes key artifacts for the ELF sample in `input/`.

In [ ]:
from pathlib import Path
import json
import subprocess

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'scripts').exists() and (p / 'input').exists():
            ROOT = p
            break

SAMPLE = sorted((ROOT / 'input').glob('*.elf'))[0]
SAMPLE

In [ ]:
subprocess.run(['python3', str(ROOT / 'scripts' / 'run_full_analysis.py')], check=True, cwd=ROOT)

In [ ]:
triage = json.loads((ROOT / 'reports' / 'json' / 'triage_report.json').read_text())
ro = json.loads((ROOT / 'reports' / 'json' / 'rodata_artifacts.json').read_text())

summary = {
    'sample_sha256': triage['sample']['sha256'],
    'file': triage['sample']['file'],
    'public_ipv4_candidates': triage['strings_summary']['public_ipv4_candidates'],
    'method_tokens': triage['strings_summary']['method_tokens'],
    'authorized_server_ip': ro['artifacts']['authorized_server_ip']['value'],
    'sigkill_cmd': ro['artifacts']['sigkill_cmd']['value'],
}
summary

In [ ]:
for row in triage['key_symbols']:
    print(f"{row['address']}  {row['name']}")

## Useful IDA Script Order

1. `ida_python/mirai_stage1_annotator.py`
2. `ida_python/mirai_string_hunt_annotator.py`

Then review `reports/disasm/` and `docs/mirai_stage_flow.md` for quick navigation anchors.